# Zadanie 2.6: mapa niedopasowania w płaszczyźnie $(r, s)$

Badamy wpływ połowy szerokości pola wiązki $r$ i przesunięcia jej środka $s$ względem guza (issue #14).
Guz startuje jako plateau o promieniu 6 w środku obszaru $L = 80$; stosujemy harmonogram bazowy
(10 frakcji co 0,4 od $t = 1$, $\beta = 1{,}5$), solver B (`cn`, $\Delta x = 0{,}2$, $\Delta t_{\max} = 0{,}05$).
Z symetrii wystarczy $s \ge 0$.

Dla każdego punktu siatki $30 \times 30$ liczymy:
- masę końcową $M(10) / M(0)$,
- najmniejszą masę $M_{\min} / M(0)$,
- czas do odrostu (pierwsza chwila powrotu $M$ do $M(0)$; brak, jeśli guz nie odrósł do $t = 10$),
- pokrycie planu $\eta$ z #16 (`observables.coverage`) w chwili odrostu (lub w $t = 10$, jeśli guz ma obszar $u \ge 0{,}5$).

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import Params, Schedule, grid, initial_condition
from offtarget.observables import coverage, min_mass, time_to_regrowth, tumor_mass
from offtarget.solvers import simulate

rng = seed_everything()

x, dx = grid(80.0, 400)
t = np.round(np.linspace(0.0, 10.0, 201), 10)
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)
u0 = initial_condition(x)
M0 = float(tumor_mass(u0, x))

r_vals = np.linspace(1.0, 24.0, 30)
s_vals = np.linspace(0.0, 15.0, 30)

t0 = time.perf_counter()
M_end = np.empty((len(s_vals), len(r_vals)))
M_min = np.empty_like(M_end)
T_reg = np.full_like(M_end, np.nan)
ETA = np.full_like(M_end, np.nan)
rows = []

for i, s in enumerate(s_vals):
    for j, r in enumerate(r_vals):
        p = Params(beta=1.5, s=float(s), r=float(r))
        u = simulate(p, schedule, x, t, u0=u0, method="cn", dose="pulse", dt_max=0.05)
        M = tumor_mass(u, x)
        m_min, _ = min_mass(t, M)
        t_reg = time_to_regrowth(t, M)
        k_eta = int(np.argmin(np.abs(t - t_reg))) if np.isfinite(t_reg) else -1
        eta = coverage(x, u[k_eta], p)
        M_end[i, j] = M[-1] / M0
        M_min[i, j] = m_min / M0
        T_reg[i, j] = t_reg
        ETA[i, j] = eta
        rows.append({
            "r": float(r),
            "s": float(s),
            "M_end_rel": float(M[-1] / M0),
            "M_min_rel": float(m_min / M0),
            "czas_odrostu": float(t_reg),
            "eta": float(eta),
        })

elapsed = time.perf_counter() - t0
df = pd.DataFrame(rows)
df.to_csv(TABLES / "zad2_06_mapa.csv", index=False)
print(f"900 symulacji w {elapsed:.1f} s; brak odrostu do t=10 w {int(np.isnan(T_reg).sum())}/900 punktach")

## Mapy czterech obserwabli w płaszczyźnie $(r, s)$

Biała ciągła linia oznacza granicę **zanik / odrost** ($M(10) = M(0)$, czyli brak powrotu masy do $M(0)$ w horyzoncie $T = 10$).
Przerywana prosta $r = s + 6$ to warunek geometrycznego objęcia startowego guza $[-6, 6]$ przez wiązkę $[s - r, s + r]$.
Krzyżykami zaznaczono trzy scenariusze z `zad2_02_scenariusze` dla $\beta = 1{,}5$.

In [ ]:
R_grid, S_grid = np.meshgrid(r_vals, s_vals)
markers = [
    (20.0, 0.0, "pełne"),
    (3.0, 0.0, "za wąskie"),
    (8.0, 6.0, "przesunięte"),
]

fig, axes = plt.subplots(2, 2, figsize=(12.2, 9.4), sharex=True, sharey=True)
panels = [
    (axes[0, 0], np.log10(np.maximum(M_end, 1e-5)), "log₁₀(M(10) / M(0))", "magma_r", (-5, 0.7)),
    (axes[0, 1], np.log10(np.maximum(M_min, 1e-5)), "log₁₀(M_min / M(0))", "viridis", (-5, 0.05)),
    (axes[1, 0], T_reg, "czas do odrostu (białe = brak do t = 10)", "plasma_r", (0, 10)),
    (axes[1, 1], ETA, "pokrycie odrostu η (białe = brak V_REC)", "coolwarm", (0, 1)),
]

for ax, Z, title, cmap_name, (vmin, vmax) in panels:
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad("white")
    im = ax.pcolormesh(R_grid, S_grid, np.ma.masked_invalid(Z), shading="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=ax, pad=0.02)
    cs = ax.contour(R_grid, S_grid, M_end, levels=[1.0], colors=["white" if "magma" in cmap_name or "viridis" in cmap_name else "black"], linewidths=2.2)
    ax.clabel(cs, fmt={1.0: "M(10)=M(0)"}, fontsize=8)
    ax.contour(R_grid, S_grid, M_end, levels=[0.1], colors=["cyan"], linestyles="--", linewidths=1.3)
    s_line = np.linspace(0, 14, 100)
    line_col = "w:" if "magma" in cmap_name else "k:"
    ax.plot(s_line + 6.0, s_line, line_col, lw=1.4, label="r = s + 6 (guz startowy)")
    for rm, sm, lbl in markers:
        ax.plot(rm, sm, "wx", ms=7, mew=2.0)
        ax.plot(rm, sm, "kx", ms=7, mew=1.0)
        dx_txt = -1.6 if rm > 17 else 0.6
        dy_txt = 0.8 if sm < 1 else 0.6
        ax.text(rm + dx_txt, sm + dy_txt, lbl, fontsize=8, weight="bold",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", alpha=0.75, ec="none"))
    ax.set(title=title, xlim=(r_vals[0], r_vals[-1]), ylim=(s_vals[0], s_vals[-1]))

for ax in axes[1]:
    ax.set_xlabel("połowa szerokości wiązki r")
for ax in axes[:, 0]:
    ax.set_ylabel("przesunięcie środka wiązki s")
axes[0, 0].legend(loc="upper left", fontsize=8)
fig.suptitle("Mapa niedopasowania w płaszczyźnie (r, s): β = 1,5, 10 frakcji", fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES / "zad2_06_mapa_rs.png", dpi=160)

## Przekrój granicy zanik / odrost

Dla wybranych przesunięć $s$ wypisujemy najmniejszą szerokość wiązki $r_{\text{kryt}}(s)$ potrzebną do utrzymania $M(10) < M(0)$
oraz nadmiarowy margines $r_{\text{kryt}}(s) - (s + 6)$ ponad sam promień guza na starcie.

In [ ]:
crit_rows = []
for i in range(0, len(s_vals), 5):
    s = s_vals[i]
    ok = np.flatnonzero(M_end[i] < 1.0)
    r_crit = float(r_vals[ok[0]]) if len(ok) else np.nan
    ok10 = np.flatnonzero(M_end[i] < 0.1)
    r_sup = float(r_vals[ok10[0]]) if len(ok10) else np.nan
    crit_rows.append({
        "s": round(float(s), 2),
        "r geometryczne (s + 6)": round(float(s + 6.0), 2),
        "r_kryt (M(10) < M(0))": round(r_crit, 2) if np.isfinite(r_crit) else np.nan,
        "margines (r_kryt - s - 6)": round(r_crit - s - 6.0, 2) if np.isfinite(r_crit) else np.nan,
        "r silnego zaniku (M(10) < 0,1 M(0))": round(r_sup, 2) if np.isfinite(r_sup) else np.nan,
    })
crit_df = pd.DataFrame(crit_rows)
crit_df.to_csv(TABLES / "zad2_06_mapa_granica.csv", index=False)
crit_df.style.format(precision=2)

## Wnioski

- **Samo geometryczne pokrycie guza startowego ($r = s + 6$) nie wystarcza.** Na całej prostej $r = s + 6$
  guz odrasta przed $t = 10$, ponieważ ogon dyfuzyjny i ekspansja frontu ($v \approx 2\sqrt{D\rho} = 2$) przed leczeniem
  i między frakcjami wynoszą komórki poza pierwotny promień 6.
- **Granica „zanik / odrost” ($M(10) = M(0)$) wymaga szerokiego marginesu.** Dla przesuniętych wiązek ($s \ge 3$)
  granica biegnie równolegle do $r = s + 6$ z dodatkowym marginesem ok. **+6,2** ($r_{\text{kryt}} \approx s + 12{,}2$).
  Przy wycentrowanej wiązce ($s = 0$) uciekają jednocześnie oba brzegi guza, więc potrzebny jest większy promień
  ($r_{\text{kryt}} \approx 14{,}1$, margines $+8{,}1$), co widać jako zaokrąglenie krzywej przy $s \to 0$.
- **Silny zanik ($M(10) < 0{,}1\,M(0)$, przerywana linia cyjanowa)** wymaga jeszcze większego marginesu
  ($r \gtrsim 16{,}7$ dla $s = 0$ i $r \gtrsim s + 15{,}5$ dla przesunięć).
- **Pokrycie odrostu $\eta$ potwierdza mechanizm off-target.** Przy dużej dawce lokalnej ($\beta = 1{,}5$) niemal w całym
  obszarze odrostu $\eta < 0{,}35$ — guz odrasta spoza pola wiązki, a nie ze środka. W strefie silnego zaniku obszar
  $V_{\text{REC}}$ ($u \ge 0{,}5$) jest pusty ($\eta = \text{NaN}$, białe pole).